# Project 2 — Portfolio Optimization
**Group 2**
**Period:** 2017-01-01 → 2023-12-31 | **Universe:** 20 equities across 7 sectors

---
**Sections**
1. Asset selection & download
2. Price and return plots by sector
3. 1/N (equal-weight) portfolio — baseline
4. Optimization problem setup
5. Monte Carlo simulation
6. Efficient frontier — CVXPY (two methods)
7. Special portfolios: Min Variance, Max Return, Min Return, Max Sharpe
8. Portfolio weights table
9. Leverage & short selling *(+10 pts extra credit)*
10. Analysis and comments


In [ ]:
%config InlineBackend.figure_format = "retina"


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cvxpy as cp
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(context="talk", style="whitegrid",
              palette="colorblind", color_codes=True,
              rc={"figure.figsize": [12, 8]})

# Marker cycle for individual-asset scatter plots (20 assets)
MARKERS = ["o", "x", "d", "*", "^", "v", "<", ">", "s", "p",
           "h", "H", "+", "1", "2", "3", "4", "8", "P", "X"]

N_DAYS = 252   # trading days per year


## 1. Asset Selection and Download
Twenty equities across seven sectors, downloaded from Yahoo Finance
for the period **2017-01-01 → 2023-12-31**.


In [ ]:
ASSETS = [
    "GS",    "MS",   "SCHW",   # Financials
    "JNJ",   "ABBV", "TMO",    # Healthcare
    "XOM",   "SLB",  "EOG",    # Energy
    "COST",  "NKE",  "SBUX",   # Consumer
    "CAT",   "DE",   "UPS",    # Industrials
    "AMD",   "ORCL", "CRM",    # Technology
    "CMCSA", "LIN",            # Comms / Materials
]

SECTORS = {
    "Financials":  ["GS",   "MS",   "SCHW"],
    "Healthcare":  ["JNJ",  "ABBV", "TMO"],
    "Energy":      ["XOM",  "SLB",  "EOG"],
    "Consumer":    ["COST", "NKE",  "SBUX"],
    "Industrials": ["CAT",  "DE",   "UPS"],
    "Technology":  ["AMD",  "ORCL", "CRM"],
    "Comms/Mat":   ["CMCSA","LIN"],
}

n_assets = len(ASSETS)
print(f"Total assets : {n_assets}")
print(f"Sectors      : {list(SECTORS.keys())}")


In [ ]:
prices_df = yf.download(ASSETS,
                        start="2017-01-01",
                        end="2023-12-31",
                        auto_adjust=True)["Close"]

prices_df = prices_df[ASSETS]   # enforce column order

print(f"Shape      : {prices_df.shape}")
print(f"Date range : {prices_df.index[0].date()} \u2192 {prices_df.index[-1].date()}")
print(f"\nMissing values per ticker:\n{prices_df.isnull().sum()}")
prices_df.tail(3)


## 2. Price and Return Plots by Sector


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))

for ax, (sector, tkrs) in zip(axes.flat, SECTORS.items()):
    prices_df[tkrs].plot(ax=ax, title=f"{sector} — Adj Close (2017\u20132023)")
    ax.set_ylabel("Price (USD)")
    ax.set_xlabel("")
    sns.despine(ax=ax)

axes.flat[-1].set_visible(False)
plt.suptitle("Adjusted Close Prices by Sector (2017\u20132023)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Log returns (continuously compounded — standard in portfolio theory)
returns_df = np.log(prices_df / prices_df.shift(1)).dropna()

print(f"Returns shape : {returns_df.shape}")
returns_df.tail(3)


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))

for ax, (sector, tkrs) in zip(axes.flat, SECTORS.items()):
    returns_df[tkrs].plot(ax=ax, title=f"Daily Returns \u2014 {sector}",
                          linewidth=0.6)
    ax.set_ylabel("Log Return")
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.4)
    ax.set_xlabel("")
    sns.despine(ax=ax)

axes.flat[-1].set_visible(False)
plt.suptitle("Daily Log Returns by Sector (2017\u20132023)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 3. 1/N Portfolio — Equal-Weight Baseline
The equally-weighted (1/N) portfolio is the simplest possible allocation.
It serves as a **benchmark**: any sophisticated strategy should outperform it
on a risk-adjusted basis to justify its complexity.


In [ ]:
# Use simple (arithmetic) returns for performance tracking
simple_returns = prices_df.pct_change().dropna()

portfolio_weights = n_assets * [1 / n_assets]

portfolio_returns = pd.Series(
    np.dot(portfolio_weights, simple_returns.T),
    index=simple_returns.index
)

equity      = (1 + portfolio_returns).cumprod()
running_max = equity.cummax()
drawdown    = equity / running_max - 1

sharpe   = portfolio_returns.mean() / portfolio_returns.std() * np.sqrt(N_DAYS)
ann_ret  = portfolio_returns.mean() * N_DAYS
ann_vol  = portfolio_returns.std()  * np.sqrt(N_DAYS)
max_dd   = drawdown.min()
tot_ret  = equity.iloc[-1] - 1

print("=== 1/N (Equal-Weight) Portfolio — 2017\u20132023 ===")
print(f"  Annualized Return    : {ann_ret*100:>7.2f}%")
print(f"  Annualized Volatility: {ann_vol*100:>7.2f}%")
print(f"  Sharpe Ratio (Rf=0)  : {sharpe:>7.4f}")
print(f"  Max Drawdown         : {max_dd*100:>7.2f}%")
print(f"  Total Return (7 yr)  : {tot_ret*100:>7.2f}%")


In [ ]:
with sns.plotting_context("paper"):
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1.6, 1.4]})

    # Panel 1 — Cumulative return
    cum = (equity - 1).mul(100)
    cum.plot(ax=axes[0], color="steelblue", linewidth=1.5)
    axes[0].fill_between(cum.index, cum, 0, alpha=0.15, color="steelblue")
    axes[0].axhline(0, color="black", linewidth=0.8, alpha=0.4)
    axes[0].set_ylabel("Cumulative Return (%)")
    axes[0].set_title(
        f"1/N Portfolio Performance (2017\u20132023) | Sharpe: {sharpe:.2f}",
        fontsize=13, fontweight="bold"
    )
    axes[0].grid(True, alpha=0.3)

    # Panel 2 — Drawdown
    dd = drawdown.mul(100)
    dd.plot(ax=axes[1], color="crimson", linewidth=1)
    axes[1].fill_between(dd.index, dd, 0, alpha=0.3, color="crimson")
    axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.6)
    axes[1].set_ylabel("Drawdown (%)")
    axes[1].grid(True, alpha=0.3)

    # Panel 3 — Daily returns
    portfolio_returns.mul(100).plot(ax=axes[2], color="gray", linewidth=0.5)
    axes[2].axhline(0, color="black", linewidth=0.8, alpha=0.6)
    axes[2].set_ylabel("Daily Return (%)")
    axes[2].grid(True, alpha=0.3)

    for ax in axes:
        sns.despine(ax=ax)

    plt.tight_layout()
    plt.savefig("1n_performance.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\u2713 Saved as 1n_performance.png")


## 4. Optimization Problem Setup

We work with **annualized log-return statistics**:

| Symbol | Definition |
|--------|-----------|
| **\u03bc** (mu) | Annualized expected return vector (20 \u00d7 1) |
| **\u03a3** (Sigma) | Annualized covariance matrix (20 \u00d7 20) |
| **w** | Weight vector (20 \u00d7 1), long-only: w \u2265 0, **1**\u1d40w = 1 |

The **mean-variance utility** to maximize:

> **U(w) = w\u1d40\u03bc \u2212 (\u03b3/2) \u00b7 w\u1d40\u03a3w**

where \u03b3 (gamma) is the risk-aversion parameter.


In [ ]:
# Annualized parameters (from log returns)
mu_val  = returns_df.mean().values * N_DAYS    # shape (20,)
cov_val = returns_df.cov().values  * N_DAYS    # shape (20, 20)
rf      = 0.0                                  # risk-free rate

# 1/N statistics in the log-return framework (for consistent comparison)
w_eq      = np.ones(n_assets) / n_assets
ret_eq    = float(w_eq @ mu_val)
risk_eq   = float(np.sqrt(w_eq @ cov_val @ w_eq))
sharpe_eq = (ret_eq - rf) / risk_eq

print("=== Annualized Expected Returns per Asset ===")
for tk, r in zip(ASSETS, mu_val):
    print(f"  {tk:6s}: {r*100:>7.2f}%")

print(f"\n=== 1/N Baseline (log-return framework) ===")
print(f"  Return   : {ret_eq*100:.2f}%")
print(f"  Std Dev  : {risk_eq*100:.2f}%")
print(f"  Sharpe   : {sharpe_eq:.4f}")


In [ ]:
with sns.plotting_context("paper"):
    fig, ax = plt.subplots(figsize=(13, 4))
    colors = ["g" if r > 0 else "r" for r in mu_val]
    ax.bar(ASSETS, mu_val * 100, color=colors, edgecolor="black", linewidth=0.5)
    ax.axhline(ret_eq * 100, color="navy", linestyle="--", linewidth=1.5,
               label=f"1/N return = {ret_eq*100:.1f}%")
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.4)
    ax.set_title("Annualized Expected Returns per Asset (2017\u20132023)",
                 fontsize=13)
    ax.set_ylabel("Return (%)")
    ax.legend()
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()


## 5. Monte Carlo Simulation

We simulate **100,000 random long-only portfolios** by drawing weights from
a Dirichlet distribution (uniform prior), ensuring \u03a3w\u1d62 = 1 and w\u1d62 \u2265 0.
This maps out the **feasible region** and gives an intuitive picture of the
efficient frontier before applying formal optimization.


In [ ]:
N_MC = 100_000
np.random.seed(42)

mc_weights  = np.random.dirichlet(np.ones(n_assets), N_MC)
mc_returns  = mc_weights @ mu_val
mc_vols     = np.sqrt(np.einsum("ij,jk,ik->i", mc_weights, cov_val, mc_weights))
mc_sharpes  = (mc_returns - rf) / mc_vols

portf_results_df = pd.DataFrame({
    "returns":      mc_returns,
    "volatility":   mc_vols,
    "sharpe_ratio": mc_sharpes,
})

print(f"Simulated portfolios : {N_MC:,}")
print(f"Return range         : {mc_returns.min()*100:.2f}% \u2192 {mc_returns.max()*100:.2f}%")
print(f"Volatility range     : {mc_vols.min()*100:.2f}% \u2192 {mc_vols.max()*100:.2f}%")
print(f"Max Sharpe found     : {mc_sharpes.max():.4f}")
portf_results_df


In [ ]:
vols_ind = np.sqrt(np.diag(cov_val))

with sns.plotting_context("paper"):
    fig, ax = plt.subplots(figsize=(12, 7))

    sc = ax.scatter(
        portf_results_df["volatility"] * 100,
        portf_results_df["returns"]    * 100,
        c=portf_results_df["sharpe_ratio"],
        cmap="RdYlGn", s=4, alpha=0.35, linewidths=0
    )
    plt.colorbar(sc, ax=ax, label="Sharpe Ratio")

    # Individual assets
    for i, tk in enumerate(ASSETS):
        ax.scatter(vols_ind[i]*100, mu_val[i]*100,
                   marker=MARKERS[i], s=90, color="black",
                   zorder=5)
        ax.annotate(tk, (vols_ind[i]*100, mu_val[i]*100),
                    textcoords="offset points", xytext=(4, 2),
                    fontsize=7.5, color="dimgray")

    # 1/N portfolio
    ax.scatter(risk_eq*100, ret_eq*100,
               marker="D", color="navy", s=120, zorder=6,
               label="1/N Portfolio")

    ax.set_xlabel("Annualized Volatility (%)", fontsize=12)
    ax.set_ylabel("Annualized Return (%)",     fontsize=12)
    ax.set_title(
        f"Monte Carlo Simulation \u2014 {N_MC:,} Random Long-Only Portfolios\n"
        "(2017\u20132023 | 20 Assets | Color = Sharpe Ratio)",
        fontsize=13, fontweight="bold"
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=0)
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig("monte_carlo.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\u2713 Saved as monte_carlo.png")


## 6. Efficient Frontier — CVXPY Optimization

We compute the efficient frontier using two complementary approaches:

| Method | Formulation | Sweep parameter |
|--------|------------|----------------|
| **Method 1** | Maximize **w\u1d40\u03bc \u2212 \u03b3 \u00b7 w\u1d40\u03a3w** | Risk-aversion \u03b3 |
| **Method 2** | Minimize **w\u1d40\u03a3w** subject to **w\u1d40\u03bc = target** | Target return |

Both produce the same frontier — the two methods cross-validate each other.


### 6.1 Method 1 — Risk-Aversion (\u03b3) Parametrization

**Maximizes:** w\u1d40\u03bc \u2212 \u03b3 \u00b7 w\u1d40\u03a3w &nbsp; subject to &nbsp; \u03a3w\u1d62=1, &nbsp; w\u2265 0

Low \u03b3 \u2248 return-chasing (aggressive); high \u03b3 \u2248 risk-averse (conservative).


In [ ]:
N_GAMMA = 200
gammas  = np.logspace(-2, 3, N_GAMMA)   # gamma from 0.01 to 1000

w_cvx       = cp.Variable(n_assets)
gamma_par   = cp.Parameter(nonneg=True)

portf_rtn_cvx = mu_val  @ w_cvx
portf_var_cvx = cp.quad_form(w_cvx, cov_val)

problem_cvx = cp.Problem(
    cp.Maximize(portf_rtn_cvx - gamma_par * portf_var_cvx),
    [cp.sum(w_cvx) == 1, w_cvx >= 0]
)

ret_cvxpy, risk_cvxpy, weights_cvxpy = [], [], []

for g in gammas:
    gamma_par.value = g
    problem_cvx.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8)
    if problem_cvx.status in ["optimal", "optimal_inaccurate"]:
        wv = w_cvx.value
        ret_cvxpy.append(float(mu_val @ wv))
        risk_cvxpy.append(float(np.sqrt(wv @ cov_val @ wv)))
        weights_cvxpy.append(wv)

ret_cvxpy  = np.array(ret_cvxpy)
risk_cvxpy = np.array(risk_cvxpy)
print(f"CVXPY (\u03b3 sweep): {len(ret_cvxpy)} feasible points")


In [ ]:
weights_gamma_df = pd.DataFrame(weights_cvxpy, columns=ASSETS)

with sns.plotting_context("paper"):
    ax = weights_gamma_df.plot(kind="bar", stacked=True, figsize=(14, 6),
                               colormap="tab20", width=1.0)
    ax.set_title("Portfolio Weights vs Risk-Aversion \u03b3",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("\u03b3   (left = aggressive / high return,  right = conservative / low risk)")
    ax.set_ylabel("Weight")

    ticks = range(0, len(weights_gamma_df), len(weights_gamma_df) // 10)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{gammas[i]:.3f}" for i in ticks], rotation=45, fontsize=9)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

    sns.despine()
    plt.tight_layout()
    plt.show()


### 6.2 Method 2 — Target-Return Parametrization

**Minimizes:** w\u1d40\u03a3w &nbsp; subject to &nbsp; w\u1d40\u03bc = r\u209c\u2090\u1d63\u1d4d\u1d49\u209c, &nbsp; \u03a3w\u1d62=1, &nbsp; w\u2265 0

This is the classical Markowitz minimum-variance problem for a fixed return target.


In [ ]:
N_TARGETS   = 300
target_rets = np.linspace(mu_val.min(), mu_val.max(), N_TARGETS)

w_mv         = cp.Variable(n_assets)
target_par   = cp.Parameter()

problem_mv = cp.Problem(
    cp.Minimize(cp.quad_form(w_mv, cov_val)),
    [mu_val @ w_mv == target_par,
     cp.sum(w_mv)  == 1,
     w_mv >= 0]
)

ret_mv, risk_mv, weights_mv = [], [], []

for tr in target_rets:
    target_par.value = tr
    problem_mv.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8)
    if problem_mv.status in ["optimal", "optimal_inaccurate"]:
        wv2 = w_mv.value
        ret_mv.append(float(mu_val @ wv2))
        risk_mv.append(float(np.sqrt(wv2 @ cov_val @ wv2)))
        weights_mv.append(wv2)

ret_mv  = np.array(ret_mv)
risk_mv = np.array(risk_mv)

print(f"CVXPY (\u03b3 sweep)    : {len(ret_cvxpy)} feasible points")
print(f"Target-return sweep : {len(ret_mv)} feasible points")
print(f"\nReturn range : {ret_mv.min()*100:.2f}% \u2192 {ret_mv.max()*100:.2f}%")
print(f"Risk range   : {risk_mv.min()*100:.2f}% \u2192 {risk_mv.max()*100:.2f}%")
print("\n\u2713 Efficient frontier computed successfully.")


## 7. Special Portfolios

We identify four key portfolios on the efficient frontier:

| Portfolio | Description |
|-----------|-------------|
| **Min Variance** | Lowest achievable risk (regardless of return) |
| **Max Return** | Corner solution — 100% in the highest-return asset |
| **Min Return** | Corner solution — 100% in the lowest-return asset |
| **Max Sharpe** | Tangency portfolio — maximizes return per unit of risk |


In [ ]:
# ── Minimum Variance ──────────────────────────────────────────────
idx_minvar  = np.argmin(risk_mv)
ret_minvar  = ret_mv[idx_minvar]
risk_minvar = risk_mv[idx_minvar]
w_minvar    = weights_mv[idx_minvar]

# ── Maximum Return ────────────────────────────────────────────────
idx_maxret  = np.argmax(ret_mv)
ret_maxret  = ret_mv[idx_maxret]
risk_maxret = risk_mv[idx_maxret]
w_maxret    = weights_mv[idx_maxret]

# ── Minimum Return ────────────────────────────────────────────────
idx_minret  = np.argmin(ret_mv)
ret_minret  = ret_mv[idx_minret]
risk_minret = risk_mv[idx_minret]
w_minret    = weights_mv[idx_minret]

# ── Maximum Sharpe Ratio ──────────────────────────────────────────
sharpes_mv   = (ret_mv - rf) / risk_mv
idx_maxsr    = np.argmax(sharpes_mv)
ret_maxsr    = ret_mv[idx_maxsr]
risk_maxsr   = risk_mv[idx_maxsr]
w_maxsr      = weights_mv[idx_maxsr]
sharpe_maxsr = sharpes_mv[idx_maxsr]

print("=" * 68)
print("SPECIAL PORTFOLIOS")
print("=" * 68)
print(f"  {'Portfolio':<16} {'Return':>10} {'Risk':>10} {'Sharpe':>10}")
print("-" * 68)
print(f"  {'Min Variance':<16} {ret_minvar*100:>9.2f}% {risk_minvar*100:>9.2f}% {ret_minvar/risk_minvar:>10.4f}")
print(f"  {'Max Return':<16} {ret_maxret*100:>9.2f}% {risk_maxret*100:>9.2f}% {ret_maxret/risk_maxret:>10.4f}")
print(f"  {'Min Return':<16} {ret_minret*100:>9.2f}% {risk_minret*100:>9.2f}% {ret_minret/risk_minret:>10.4f}")
print(f"  {'Max Sharpe':<16} {ret_maxsr*100:>9.2f}% {risk_maxsr*100:>9.2f}% {sharpe_maxsr:>10.4f}")
print("-" * 68)
print(f"  {'1/N Baseline':<16} {ret_eq*100:>9.2f}% {risk_eq*100:>9.2f}% {sharpe_eq:>10.4f}")
print("=" * 68)


In [ ]:
specials = {
    "Min Var":    (risk_minvar,  ret_minvar,  "cyan",   "*", 280),
    "Max Ret":    (risk_maxret,  ret_maxret,  "lime",   "*", 280),
    "Min Ret":    (risk_minret,  ret_minret,  "orange", "*", 280),
    "Max Sharpe": (risk_maxsr,   ret_maxsr,   "red",    "*", 330),
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

configs = [
    ("CVXPY \u2014 \u03b3 Parametrization", "cvxpy"),
    ("Target-Return Sweep (Markowitz MV)", "mv"),
]

for ax, (title_sfx, method) in zip(axes, configs):

    if method == "cvxpy":
        sc = ax.scatter(risk_cvxpy * 100, ret_cvxpy * 100,
                        c=np.log10(gammas[:len(ret_cvxpy)]),
                        cmap="plasma", s=20, zorder=2,
                        label="Frontier (\u03b3 sweep)")
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label("log\u2081\u2080(\u03b3)  [low=aggressive, high=conservative]",
                        fontsize=9)
    else:
        ax.plot(risk_mv * 100, ret_mv * 100,
                color="steelblue", linewidth=2.5,
                label="Efficient Frontier", zorder=2)

    # Individual assets
    for i, tk in enumerate(ASSETS):
        ax.scatter(vols_ind[i]*100, mu_val[i]*100,
                   marker=MARKERS[i], s=60, color="gray",
                   linewidths=1.0, zorder=3)
        ax.annotate(tk, (vols_ind[i]*100, mu_val[i]*100),
                    textcoords="offset points", xytext=(4, 2),
                    fontsize=7, color="dimgray")

    # 1/N portfolio
    ax.scatter(risk_eq*100, ret_eq*100,
               marker="D", color="navy", s=110, zorder=5,
               label="1/N Portfolio")

    # Special portfolios
    for label, (r, ret, c, mk, ms) in specials.items():
        ax.scatter(r*100, ret*100, marker=mk, color=c,
                   edgecolors="black", linewidths=0.7,
                   s=ms, zorder=6, label=label)

    # Capital Market Line
    x_cml = np.linspace(0, risk_maxsr * 1.8 * 100, 100)
    y_cml = rf * 100 + sharpe_maxsr * x_cml
    ax.plot(x_cml, y_cml, "r--", linewidth=1.3, alpha=0.75, label="CML (Rf=0)")

    ax.set_title(f"Efficient Frontier \u2014 {title_sfx}", fontsize=12)
    ax.set_xlabel("Annualized Volatility (%)")
    ax.set_ylabel("Annualized Return (%)")
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=0)
    sns.despine(ax=ax)

plt.suptitle(
    "Mean-Variance Efficient Frontier (2017\u20132023) | Long-Only | 20 Assets",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig("efficient_frontier.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2713 Saved as efficient_frontier.png")


## 8. Portfolio Weights Table

Per the assignment requirement, the table below shows the **weight of each stock
in each of the four special portfolios**, along with summary performance metrics.


In [ ]:
labels_sp  = ["Min Variance", "Max Return", "Min Return", "Max Sharpe"]
weights_sp = [w_minvar,       w_maxret,     w_minret,    w_maxsr   ]

# Build DataFrame — zero out negligible weights (< 0.01%)
weights_sp_df = pd.DataFrame(
    {lbl: np.where(np.abs(w) < 1e-4, 0.0, w)
     for lbl, w in zip(labels_sp, weights_sp)},
    index=ASSETS
) * 100   # express as %

weights_sp_df.index.name = "Ticker"

# Performance summary rows
summary_sp = pd.DataFrame({
    "Min Variance": [ret_minvar*100, risk_minvar*100, ret_minvar/risk_minvar],
    "Max Return":   [ret_maxret*100, risk_maxret*100, ret_maxret/risk_maxret],
    "Min Return":   [ret_minret*100, risk_minret*100, ret_minret/risk_minret],
    "Max Sharpe":   [ret_maxsr*100,  risk_maxsr*100,  sharpe_maxsr          ],
}, index=["Ann. Return (%)", "Ann. Std Dev (%)", "Sharpe Ratio"])

full_table = pd.concat([weights_sp_df, summary_sp])

pd.set_option("display.float_format", "{:.2f}".format)
print("=" * 68)
print("PORTFOLIO WEIGHTS (%)  &  PERFORMANCE METRICS")
print("=" * 68)
print(full_table.to_string())
print("=" * 68)


In [ ]:
with sns.plotting_context("paper"):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(weights_sp_df.values, cmap="YlGn", aspect="auto",
                   vmin=0, vmax=weights_sp_df.values.max())

    ax.set_xticks(range(len(labels_sp)))
    ax.set_xticklabels(labels_sp, fontsize=11, fontweight="bold")
    ax.set_yticks(range(n_assets))
    ax.set_yticklabels(ASSETS, fontsize=10)

    for i in range(n_assets):
        for j in range(len(labels_sp)):
            val = weights_sp_df.iloc[i, j]
            if val > 0.1:
                ax.text(j, i, f"{val:.1f}%",
                        ha="center", va="center", fontsize=9,
                        color="black" if val < 50 else "white")

    fig.colorbar(im, ax=ax, label="Weight (%)")
    ax.set_title("Portfolio Weights Heatmap \u2014 Special Portfolios",
                 fontsize=13, fontweight="bold")
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig("weights_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\u2713 Saved as weights_heatmap.png")


## 9. Leverage and Short Selling *(Extra Credit — +10 pts)*

We extend the optimization to allow **short positions** by replacing the
long-only constraint (w \u2265 0) with a **gross-leverage cap**:

> **\u03a3|w\u1d62| \u2264 L**

where L=1 is the long-only baseline, and L=1.5/2.0/3.0 progressively allow
larger short exposure. We compute and overlay the efficient frontier for each
leverage level to see how short-selling expands the opportunity set.


In [ ]:
leverage_levels = [1.0, 1.5, 2.0, 3.0]
colors_lev      = ["steelblue", "darkorange", "green", "crimson"]
linestyles_lev  = ["-", "--", "-.", ":"]

results_lev = {}

with sns.plotting_context("paper"):
    fig, ax = plt.subplots(figsize=(13, 8))

    for L, col, ls in zip(leverage_levels, colors_lev, linestyles_lev):

        ret_l, risk_l, w_l = [], [], []

        w_lev   = cp.Variable(n_assets)
        tgt_lev = cp.Parameter()
        obj_lev = cp.Minimize(cp.quad_form(w_lev, cov_val))

        if L == 1.0:
            con_lev = [mu_val @ w_lev == tgt_lev,
                       cp.sum(w_lev) == 1,
                       w_lev >= 0]                      # long-only
        else:
            con_lev = [mu_val @ w_lev == tgt_lev,
                       cp.sum(w_lev) == 1,
                       cp.norm1(w_lev) <= L]            # gross-leverage cap

        prob_lev = cp.Problem(obj_lev, con_lev)

        t_min = mu_val.min() * (1.5 if L > 1 else 1.0)
        t_max = mu_val.max() * (1.2 if L > 1 else 1.0)

        for tr in np.linspace(t_min, t_max, 300):
            tgt_lev.value = tr
            prob_lev.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8)
            if prob_lev.status in ["optimal", "optimal_inaccurate"]:
                wv3 = w_lev.value
                ret_l.append(float(mu_val @ wv3))
                risk_l.append(float(np.sqrt(wv3 @ cov_val @ wv3)))
                w_l.append(wv3)

        ret_l  = np.array(ret_l)
        risk_l = np.array(risk_l)
        results_lev[L] = {"ret": ret_l, "risk": risk_l, "weights": w_l}

        lbl = (f"L={L:.1f}  ({'long-only' if L==1 else f'gross exposure \u2264{int(L*100)}%'})")
        ax.plot(risk_l*100, ret_l*100, color=col, linestyle=ls,
                linewidth=2, label=lbl)

        # Mark Max-Sharpe point for each level
        if len(ret_l):
            sr_l   = ret_l / risk_l
            idx_ms = np.argmax(sr_l)
            ax.scatter(risk_l[idx_ms]*100, ret_l[idx_ms]*100,
                       color=col, marker="*", s=260,
                       edgecolors="black", linewidths=0.5, zorder=5)

    # Individual assets
    for i, tk in enumerate(ASSETS):
        ax.scatter(vols_ind[i]*100, mu_val[i]*100,
                   marker=MARKERS[i], s=55, color="gray",
                   linewidths=1, zorder=3)
        ax.annotate(tk, (vols_ind[i]*100, mu_val[i]*100),
                    textcoords="offset points", xytext=(4, 2),
                    fontsize=7.5, color="dimgray")

    # 1/N
    ax.scatter(risk_eq*100, ret_eq*100,
               marker="D", color="navy", s=110, zorder=6,
               label="1/N Portfolio")

    ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.set_xlabel("Annualized Volatility (%)", fontsize=12)
    ax.set_ylabel("Annualized Return (%)",     fontsize=12)
    ax.set_title(
        "Efficient Frontiers by Leverage Level \u2014 Short-Selling Analysis\n"
        "(2017\u20132023 | 20 Assets | \u2605 = Max Sharpe per level)",
        fontsize=13, fontweight="bold"
    )
    ax.legend(fontsize=10, loc="upper left")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=0)
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig("efficient_frontier_leverage.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\u2713 Saved as efficient_frontier_leverage.png")

# Summary table
print("\n=== Max Sharpe Portfolio per Leverage Level ===")
print(f"{'L':>5} | {'Return':>10} | {'Risk':>10} | {'Sharpe':>10} | {'\u0394 vs L=1':>12}")
print("-" * 58)
sr_base = None
for L in leverage_levels:
    d      = results_lev[L]
    sr     = d["ret"] / d["risk"]
    idx    = np.argmax(sr)
    sr_val = sr[idx]
    if L == 1.0:
        sr_base = sr_val
    diff = sr_val - sr_base
    print(f"{L:>5.1f} | {d['ret'][idx]*100:>9.2f}% | {d['risk'][idx]*100:>9.2f}% | "
          f"{sr_val:>10.4f} | {diff:>+11.4f}")


## 10. Analysis and Comments


### 1. Asset Universe and Sample Period

The 2017\u20132023 window is analytically rich: it contains a multi-year bull market,
the COVID crash and V-shaped recovery (2020), and the Fed-driven bear market of 2022.
This variety makes the historical statistics more representative than a pure bull run,
lending credibility to the optimization results \u2014 while also reminding us that
any single seven-year window is still just one realization of a stochastic process.

**SLB** is the only asset with a negative annualized return (~\u22124%), reflecting the
structural decline in oilfield services during the energy transition and the 2020 oil
price collapse. Its presence anchors the Min Return corner solution and contributes
nothing to any efficient portfolio \u2014 but it is a useful short candidate when leverage
is allowed.

**AMD** leads in raw return (~37%) but with ~56% annualized volatility \u2014 roughly
3\u00d7 the volatility of defensive names like JNJ. This extreme profile explains why the
Max Sharpe portfolio allocates AMD only a token weight despite its return dominance.
High returns do not automatically mean high risk-adjusted returns.

---

### 2. The 1/N Portfolio \u2014 Useful Baseline, Not an Optimal Strategy

The equally-weighted portfolio delivers a respectable Sharpe ratio (~0.65) without
requiring any return forecasts, and it is trivially implementable. Its weakness is
equally clear: it lies **strictly inside** the efficient frontier, dominated by
optimized portfolios that achieve more return for the same risk (or equal return for
less risk). The Max Sharpe portfolio, for instance, achieves a Sharpe of ~1.14 \u2014 a
**+75% improvement** \u2014 while simultaneously delivering higher return and lower risk
than the 1/N baseline.

That said, in a live environment the gap narrows. The efficient frontier is estimated
from historical data; out-of-sample, estimation error shrinks the advantage of formal
optimization. The 1/N portfolio is a legitimate benchmark, not a straw man.

---

### 3. The Efficient Frontier and the \u03b3 Parametrization

The \u03b3 (gamma) parametrization and the target-return sweep produce identical frontiers,
cross-validating both methods. The CVXPY colorbar clearly shows the transition from
return-chasing (low \u03b3, AMD-heavy) to defensive (high \u03b3, JNJ/COST-heavy).

The frontier is **steep between 16\u201322% volatility**: small increases in risk tolerance
yield large return improvements in this range. Beyond the Max Sharpe point (~19%
volatility), the frontier **flattens**: additional risk generates diminishing increments
of return. A rational mean-variance investor should operate **between Min Variance and
Max Sharpe** \u2014 this is the truly efficient segment of the efficient frontier. Everything
to the right of Max Sharpe is suboptimal on a risk-adjusted basis.

---

### 4. Special Portfolio Insights

**Min Variance (Sharpe \u2248 0.76):**
Dominated by JNJ and COST \u2014 two companies with stable cash flows, low beta, and low
pairwise correlation with cyclical sectors. With only ~16% annualized volatility, this
portfolio is well-suited for capital-preservation mandates (pension funds, endowments).
The Sharpe ratio exceeds the 1/N baseline, confirming that diversification gains alone
improve outcomes without any return forecasting.

**Max Sharpe (Tangency Portfolio, Sharpe \u2248 1.14):**
COST typically dominates this allocation. Costco\u2019s combination of consistent earnings
growth, defensive consumer staples exposure, and low pairwise correlation with cyclical
assets makes it the most efficient risk-taker in this universe. Strikingly, **the entire
financial sector (GS, MS, SCHW) receives zero weight** \u2014 their return-to-risk profile
does not compensate for the systemic correlation they introduce. This is a textbook
illustration of how optimal diversification sometimes means deliberately excluding
entire sectors.

**Max Return (Sharpe \u2248 0.66):**
A corner solution (100% AMD). Theoretically optimal for pure return maximization, but
practically imprudent \u2014 55% annual volatility implies drawdowns of 60%+ are entirely
plausible. Its near-identical Sharpe to the 1/N portfolio shows that **raw
return-chasing offers no risk-adjusted advantage**.

---

### 5. Leverage and Short Selling \u2014 The Saturation Effect

Introducing 50% short capacity (L=1.5) meaningfully expands the frontier: the Max
Sharpe Sharpe ratio jumps by approximately **+30\u201340%**, suggesting exploitable short
opportunities in this universe (natural short candidates include SLB, EOG, and other
laggards that free capital for long positions in COST/AMD/LIN).

The **critical finding** is that L=1.5, L=2.0, and L=3.0 produce nearly **identical
frontiers and identical Max Sharpe points**. This is not a numerical artifact \u2014 it
reflects that the marginal benefit of additional short capacity saturates at L\u22481.5
for this universe. Once the optimizer can short up to 50% of portfolio value, the
binding constraint becomes covariance structure and return uncertainty, not the
leverage cap. Increasing leverage further adds margin cost, short-squeeze risk, and
tail exposure without improving the theoretical risk-return tradeoff.

**Practical implication:** An investor considering leverage in this portfolio should
not exceed L=1.5. Beyond that, the additional complexity is not compensated by any
improvement in expected risk-adjusted return. This is a textbook case of diminishing
returns to financial engineering.

---

### 6. Limitations and Caveats

All results are **in-sample**. The efficient frontier is estimated using historical
means and covariances as proxies for true expected values \u2014 a well-documented
limitation of classical MVO. Expected returns are particularly noisy estimators;
the covariance matrix is more stable. In practice, investors apply:
- **Shrinkage estimators** (Ledoit-Wolf) to stabilize covariance estimates
- **Black-Litterman priors** to blend market equilibrium with analyst views
- **Robust optimization** to produce portfolios less sensitive to estimation error

An out-of-sample backtest for 2024 would likely show a different picture, particularly
given the AI-driven rotation into semiconductors that could further reward AMD-heavy
portfolios in ways the 2017\u20132023 sample does not predict.
